### Запуск Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import *
import os


spark = SparkSession \
            .builder \
            .config("spark.master", "local") \
            .config('spark.executor.cores', '8')\
            .config('spark.executor.memory', '2g')\
            .config('spark.driver.memory', '2g')\
            .config('spark.dynamicAllocation.minExecutors', '4')\
            .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
            .config("spark.jars.packages", "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2") \
            .appName("spark") \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
hadoop_conf.set("fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


print("Активные Spark сессии:", 'http://localhost:4040/jobs')

In [ ]:
# Создадим dataframe словарь
# Путь к файлу в S3
s3_path = "s3a://prod/api/earthquake/*.json"
s3_path_regions = "s3a://prod/jdbc/regions/*.parquet"

# Чтение JSON-файла
df = spark.read.json(s3_path)
regions = spark.read.parquet(s3_path_regions)

# # Покажем несколько строк
# df.show(truncate=False)


flattened = df.selectExpr("explode(features) as feature") \
    .select(
        col("feature.id").alias("id"),
        from_unixtime(col("feature.properties.time") / 1000).alias("time"),
        col("time").cast("date").alias("load_date"),
        trim(split(col("feature.properties.place"), ",").getItem(0)).alias("place"),
        trim(split(col("feature.properties.place"), ",").getItem(1)).alias("initial_region"), 
        col("feature.properties.mag").alias("magnitude"),
        col("feature.properties.felt").alias("felt"),
        col("feature.properties.tsunami").alias("tsunami"),
        col("feature.properties.url").alias("url"),
        col("feature.geometry.coordinates")[0].alias("longitude"),
        col("feature.geometry.coordinates")[1].alias("latitude"),
        col("feature.geometry.coordinates")[2].alias("depth"),
        md5(lower(trim(col("feature.properties.place")))).alias("place_hash")
    )


flattened.alias("f")\
    .join(regions.alias("r"), "place_hash", "left")\
    .where("initial_region IS NULL") \
    .drop('initial_region', 'place_hash') \
    .show(truncate=False)

# felt - Сколько людей сообщили, что ощутили землетрясение (в данном случае 1 человек)
# tsunami - Флаг цунами: 1 — возможно, 0 — нет
# url - Ссылка на страницу события на сайте USGS


In [ ]:
ch_max_df = spark.read \
    .format("jdbc") \
    .option("url", 'jdbc:clickhouse://clickhouse:8123/default',) \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("dbtable", f"(SELECT MAX(load_date) as max_date FROM enriched_earthquakes)") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .load()

max_date = ch_max_df.collect()[0]["max_date"].strftime("%Y-%m-%d")

if max_date == '1970-01-01':
    predicate = "1=1"
else:
    predicate = f"load_date > '{max_date}'"

predicate

In [ ]:
spark.stop()